In [ ]:
from classes import AMDModel
import xarray as xr 
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from data_utils import *

In [ ]:
# Create a 7x7 grid with simple linear flow from west to east
n = 7
Q = np.ones((n, n), dtype = np.float32) * 1.5  # Constant flow for easier observation
ID = np.arange(n * n, dtype = np.int64).reshape((n, n))
outID = -np.ones((n, n), dtype = np.int64)
source = np.zeros((n, n), dtype = np.int16)
ore = np.zeros((n, n), dtype = np.float64)

# Only ore at (0, 0)
ore[0, 0] = 2700.0
source[0, 0] = 1

# Create linear flow path: (0,0) → (0,1) → (0,2) → ... → (0,6)
# From (i,j), flow goes to (i,j+1) (moving east along row 0)
for j in range(n - 1):
    outID[0, j] = ID[0, j + 1]

# All other cells in row 0 that don't flow east go to -1 (no outflow)
outID[0, n-1] = -1  # Last cell has no outflow

# All cells in other rows have no outflow (not part of the main flow path)
for i in range(1, n):
    for j in range(n):
        outID[i, j] = -1

ds = xr.Dataset(
    {
        "Q": (("lon", "lat"), Q),
        "ID": (("lon", "lat"), ID),
        "outID": (("lon", "lat"), outID),
        "source": (("lon", "lat"), source),
        "ore": (("lon", "lat"), ore)
    },
    coords={
        "lon": np.arange(n),
        "lat": np.arange(n)
    }
)

date_range = pd.date_range("2015-01-01", "2025-01-01", freq = "W")

Q_repeated = np.repeat(Q[np.newaxis, :, :], repeats = len(date_range), axis = 0)
Q = xr.DataArray(
    Q_repeated,
    dims = ("time", "lon", "lat"),
    coords = {
        "time": date_range,
        "lon": np.arange(n),
        "lat": np.arange(n)
    }
)

ds["Q"] = (("time", "lon", "lat"), Q.data)
ds["time"] = date_range

In [ ]:
ds

In [ ]:
plt.imshow(ds["Q"][0, :, :])
plt.colorbar();

In [ ]:
plt.imshow(ds["ore"][: , :])
plt.colorbar();

In [ ]:
model = AMDModel(ds, "week")

In [ ]:
model.run()

In [ ]:
model.dataset = xr.open_dataset("amdflow_output.nc")
ferric_iron = model.dataset["ferric_iron"] 
hydron = model.dataset["hydrogen_ion"]
sulphate = model.dataset["sulphate"]
iron_III_hydroxide = model.dataset["iron_III_hydroxide"]
ferrous_iron = model.dataset["ferrous_iron"]
model.dataset.close()

In [ ]:
ferric_iron.isel(time = [-2]).plot()

In [ ]:

np.isnan(ferric_iron.sel(lon = 2, lat = 2)).any()
np.isnan(ferric_iron.sel(lon = 0, lat = 0)).any()

In [ ]:
fig, ((ax1, ax2), (ax3, ax4), (ax5, ax6), (ax7, ax8), (ax9, ax10)) = plt.subplots(nrows=5, ncols=2, figsize=(14, 14))

ferric_iron.sel(lon = 0, lat = 0).plot(ax = ax1, label = "ferric_iron [0, 0]")
ferric_iron.sel(lon = 6, lat = 0).plot(ax = ax2, label = "ferric_iron [0, 6]")

sulphate.sel(lon = 0, lat = 0).plot(ax = ax3, label = "sulphate [0, 0]")
sulphate.sel(lon = 6, lat = 0).plot(ax = ax4, label = "sulphate [0, 6]")

ferrous_iron.sel(lon = 0, lat = 0).plot(ax = ax5, label = "ferrous_iron [0, 0]")
ferrous_iron.sel(lon = 6, lat = 0).plot(ax = ax6, label = "ferrous_iron [0, 6]")

hydron.sel(lon = 0, lat = 0).plot(ax = ax7, label = "hydron [0, 0]")
hydron.sel(lon = 6, lat = 0).plot(ax = ax8, label = "hydron [0, 6]")

iron_III_hydroxide.sel(lon = 0, lat = 0).plot(ax = ax9, label = "iron_III_hydroxide [0, 0]")
iron_III_hydroxide.sel(lon = 6, lat = 0).plot(ax = ax10, label = "iron_III_hydroxide [0, 6]")
fig.tight_layout()

In [ ]:
plt.plot(ferric_iron.sel(lon = 1, lat = 0).values)

In [ ]:
sulphate.isel(time = [200]).plot()
plt.gca().invert_yaxis()

In [ ]:
iron_III_hydroxide.isel(time = [-2]).plot()
plt.gca().invert_yaxis()

In [ ]:
ferrous_iron.isel(time = [-2]).plot()
plt.gca().invert_yaxis()

In [ ]:
hydron.isel(time = [-2]).plot()
plt.gca().invert_yaxis()

In [ ]:
animation_plot(sulphate)